# The LLC4320 grid

Reference documentation for the grid every field in this pipeline is
computed on.  Companions: [Gradients.md](../../docs/Gradients.md) (what
the differencing operators do with it) and
[Fields.md](../../docs/Fields.md) (per-channel reference).

Three things live here:

1. **Horizontally** — the Arakawa C grid, in ECCO / xmitgcm naming,
   with the tracer-gradient operation path drawn on it.
2. **Vertically** — the MITgcm column: `Z`, `Zl`, `Zu`, `Zp1`, `drF`,
   `drC`, and the sign convention that separates dbof from the model.
3. **The actual depths** — every level of the LLC_DEPTH store, read
   live from the grid (Section 3).  The schematic in Section 2 uses
   round illustrative numbers on purpose; Section 3 is the real thing.

Sections 1 and 2 need nothing but `matplotlib`.  Section 3 reads
`s3://dbof/LLC4320_RAW/DEPTH/grid.zarr`, so it needs NRP Nautilus
credentials — run it on a machine that has them.

Regenerate this notebook's skeleton with
`python notebooks/notebooks_field_validation/build_grid_notebook.py`
(that OVERWRITES saved outputs — re-run afterwards).

## Section 1 — The horizontal grid (Arakawa C)

LLC4320 is an Arakawa C-grid ([MITgcm horizontal-grid
documentation](https://mitgcm.readthedocs.io/en/latest/algorithm/horiz-grid.html)):
tracers at cell **centres** `(j, i)`, `U` on the **west** faces
`(j, i_g)`, `V` on the **south** faces `(j_g, i)`, vorticity on the
**corners** `(j_g, i_g)`.

A finite difference moves a quantity by half a cell — differencing a
centre field lands on a face, differencing a face field lands on a
centre or a corner depending on direction.  That is the whole reason
the gradient routines interpolate at all, and the reason the ORDER of
squaring and interpolating matters (see Gradients.md).

In [ ]:
# Section 1: where every horizontal quantity lives.
import matplotlib.pyplot as plt

from dbof.plotting import schematics

schematics.draw_cgrid_schematic()
plt.show()

## Section 2 — The vertical grid (topology)

Cell centres `Z` (dim `k`) carry `Theta`, `Salt`, and `U`/`V` at the
same `k`; interfaces carry `W` (dim `k_l`).  The interface arrays
overlap: `Zp1` is all `nk+1` interfaces, `Zl` is the low-index
(shallower) one of each cell, `Zu` the high-index (deeper) one.

**The `l`/`u` suffixes are about INDEX POSITION, not depth**, and that
stays true under either sign convention — the single most common source
of off-by-one errors when reading MITgcm vertical output.

**Sign convention.** MITgcm/ECCO use a positive-UPWARD `Z`, so stored
values are negative below the surface.  dbof works in positive-DOWNWARD
depth (`vertical_helpers._get_depth_coord` flips the sign and warns).
The figure is drawn in dbof depth with the native value in parentheses.

The depths below are round illustrative numbers — the point of the
figure is the topology.  For the real levels see Section 3.

In [ ]:
# Section 2: vertical topology and naming (illustrative depths).
schematics.draw_vertical_schematic()
plt.show()

## Section 3 — Every LLC4320 depth level, from the grid

Read live from `s3://dbof/LLC4320_RAW/DEPTH/grid.zarr` — the same
store the DEPTH pipeline uses (`grid_setup.set_up_grid("DEPTH", ...)`).
Only the 1-D vertical coordinates are pulled, so this is a small read;
the full grid is ~1.5 GB and is NOT loaded here.

Columns, all in metres and all POSITIVE DOWNWARD:

| column | meaning |
|---|---|
| `k` | tracer-level index |
| `Z_depth` | cell-centre depth (native `Z` = `-Z_depth`) |
| `Zl_depth` | shallower (low-index) interface of cell `k` |
| `Zu_depth` | deeper (high-index) interface of cell `k` |
| `drF` | cell thickness, `|Zu - Zl|` — the vertical-mean weight |
| `drC_to_next` | centre-to-centre distance from `k` to `k+1` |

`drC` is DERIVED here (`diff` of the centre depths): the grid store
carries `Z`, `Zl`, `Zu`, `Zp1` and `drF`, but not `drC`.

In [ ]:
# Section 3a: pull the 1-D vertical coordinates from the grid store.
import numpy as np
import pandas as pd

from dbof.global_dataset_creation.data_sources import LLC_DEPTH_SOURCE
from dbof.llc4320_ingestion.get_raw_data import get_llc_depth_gridfile

_src = dict(LLC_DEPTH_SOURCE)
grid = get_llc_depth_gridfile(
    _src["s3_endpoint"], _src["bucket"],
    _src.get("grid_folder", _src["folder"]),
)   # lazy: nothing is read until we ask for the vertical vars


def as_depth(da):
    """1-D vertical coordinate as POSITIVE-DOWNWARD depth in metres.

    Mirrors ``vertical_helpers._get_depth_coord``: MITgcm stores these
    positive-upward (negative below the surface), dbof works in depth.
    Inputs: da (xr.DataArray, 1-D).
    Outputs: np.ndarray of depths, positive downward.
    Generated by LH and Claude
    """
    v = np.asarray(da.values, dtype=float)
    return -v if np.nanmean(v) < 0 else v


vert = {name: as_depth(grid[name])
        for name in ("Z", "Zl", "Zu", "Zp1", "drF") if name in grid}
print({k: v.shape for k, v in vert.items()})

In [ ]:
# Section 3b: assemble the per-level table.
nk = vert["Z"].size
_drc = np.full(nk, np.nan)
_drc[:-1] = np.diff(vert["Z"])          # centre-to-centre, derived

levels = pd.DataFrame({
    "k": np.arange(nk),
    "Z_depth": vert["Z"],
    "Zl_depth": vert["Zl"][:nk] if "Zl" in vert else np.nan,
    "Zu_depth": vert["Zu"][:nk] if "Zu" in vert else np.nan,
    "drF": vert["drF"][:nk] if "drF" in vert else np.nan,
    "drC_to_next": _drc,
}).set_index("k")

print(f"{nk} levels; surface centre at {levels.Z_depth.iloc[0]:.3f} m, "
      f"deepest centre at {levels.Z_depth.iloc[-1]:.3f} m")
if "drF" in vert:
    print(f"column thickness (sum drF): {vert['drF'][:nk].sum():.2f} m")
with pd.option_context("display.max_rows", None,
                       "display.float_format", "{:.3f}".format):
    display(levels)

## Section 4 — Write the documentation artifacts

`docs/` is plain markdown (Sphinx + MyST, no notebook renderer), so the
docs page gets PNGs plus a generated table rather than this notebook.
This cell writes:

- `docs/images/grid_cgrid_schematic.png`
- `docs/images/grid_vertical_schematic.png`
- `docs/images/sparkle_cancellation.png`
- the depth table, into `docs/Grid.md` between its
  `<!-- BEGIN depth-levels -->` / `<!-- END depth-levels -->` markers

The PNGs alone can be regenerated without this notebook (and without
credentials) by running
`python -m dbof.plotting.schematics --outdir docs/images`.

In [ ]:
# Section 4: refresh docs/images/*.png and the docs/Grid.md table.
from pathlib import Path

BEGIN = "<!-- BEGIN depth-levels (generated by Grid.ipynb) -->"
END = "<!-- END depth-levels -->"


def repo_root(start=None):
    """Walk up from *start* to the directory holding pyproject.toml.

    Inputs: start (Path or None; defaults to the working directory).
    Outputs: Path of the repo root.
    Generated by LH and Claude
    """
    here = Path(start or Path.cwd()).resolve()
    for cand in (here, *here.parents):
        if (cand / "pyproject.toml").exists():
            return cand
    raise FileNotFoundError("no pyproject.toml above " + str(here))


def md_table(df, floatfmt="{:.3f}"):
    """Render a DataFrame as a GitHub/MyST markdown table.

    Hand-rolled so the docs build needs no ``tabulate`` dependency.
    Inputs: df (pd.DataFrame, index is written as its own column);
    floatfmt (format string for float cells).
    Outputs: str.
    Generated by LH and Claude
    """
    cols = [df.index.name or "index", *df.columns]
    out = ["| " + " | ".join(cols) + " |",
           "|" + "|".join(["---"] * len(cols)) + "|"]
    for idx, row in df.iterrows():
        vals = [str(idx)]
        for v in row:
            vals.append("" if pd.isna(v) else (
                floatfmt.format(v) if isinstance(v, float) else str(v)))
        out.append("| " + " | ".join(vals) + " |")
    return "\n".join(out)


def inject(path, body, begin=BEGIN, end=END):
    """Replace the text between two markers in a file, in place.

    Inputs: path (Path); body (str to place between the markers);
    begin/end (marker lines, which are preserved).
    Outputs: Path written.
    Generated by LH and Claude
    """
    text = Path(path).read_text()
    i, j = text.find(begin), text.find(end)
    if i < 0 or j < 0:
        raise ValueError(f"markers not found in {path}")
    new = text[:i] + begin + "\n\n" + body + "\n\n" + text[j:]
    Path(path).write_text(new)
    return Path(path)


root = repo_root()
for _p in schematics.save_all(root / "docs" / "images"):
    print(f"wrote {_p.relative_to(root)}")

# The PNGs need no data; the table only exists if Section 3 ran.
if "levels" in dir():
    _caption = (f"{nk} levels, read from "
                f"`{_src['bucket'].rstrip('/')}/"
                f"{_src.get('grid_folder', _src['folder'])}"
                "/grid.zarr`.  All values in metres, POSITIVE "
                "DOWNWARD (native MITgcm `Z` is the negative of "
                "`Z_depth`).  `drC_to_next` is derived from the "
                "centre depths.")
    _doc = inject(root / "docs" / "Grid.md",
                  _caption + "\n\n" + md_table(levels))
    print(f"wrote {_doc.relative_to(root)}")
else:
    print("Section 3 has not run -- docs/Grid.md table left as is.")

## Where to go next

- **What the operators do with this grid** —
  [docs/Gradients.md](../../docs/Gradients.md), including the sparkle
  schematic (why the order of squaring and interpolating matters).
- **The A/B evidence for the square-first fix** —
  [field_validation_sparkle.ipynb](field_validation_sparkle.ipynb).
- **Per-channel definitions** —
  [docs/Fields.md](../../docs/Fields.md).
- **Vertical reductions** (`_sfc`, `_z25m`, `_mld`, `_mld_mean`) —
  `dbof.preprocessing.vertical_helpers`, whose depth handling is the
  convention drawn in Section 2.